In [1]:
import cv2
import numpy as np

In [2]:
src_images = ['Images/Augment/01-sample.png',
              'Images/Augment/02-Resistor.png',
             'Images/Augment/03-Battery.png']

In [3]:
def aruco_display(corners, ids, rejected, frame):
    image = frame.copy()
    ids = ids.flatten()
    
    for (markerCorner, markerID) in zip(corners, ids):
            
            corners = markerCorner.reshape((4, 2))
            (topLeft, topRight, bottomRight, bottomLeft) = corners
            
            topRight = (int(topRight[0]), int(topRight[1]))
            bottomRight = (int(bottomRight[0]), int(bottomRight[1]))
            bottomLeft = (int(bottomLeft[0]), int(bottomLeft[1]))
            topLeft = (int(topLeft[0]), int(topLeft[1]))

            cv2.line(image, topLeft, topRight, (0, 255, 0), 2)
            cv2.line(image, topRight, bottomRight, (0, 255, 0), 2)
            cv2.line(image, bottomRight, bottomLeft, (0, 255, 0), 2)
            cv2.line(image, bottomLeft, topLeft, (0, 255, 0), 2)
            
            cX = int((topLeft[0] + bottomRight[0]) / 2.0)
            cY = int((topLeft[1] + bottomRight[1]) / 2.0)
            cv2.circle(image, (cX, cY), 4, (0, 0, 255), -1)
            
            cv2.putText(image, str(markerID),(topLeft[0], topLeft[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 2)
            # print("[Inference] ArUco marker ID: {}".format(markerID))
            
    return image

In [10]:
def float_image(frame, corners, ids, rejected, matrix_coefficients, distortion_coefficients):
    h, w = frame.shape[:2]
    mask = np.zeros([h, w], dtype=np.uint8)
    warp_image = np.zeros_like(frame)

    for (markerCorner, markerID) in zip(corners, ids):
        try:
            rvec, tvec, markerPoints = cv2.aruco.estimatePoseSingleMarkers(
                markerCorner, 5, matrix_coefficients, distortion_coefficients)

            src_image = cv2.imread(src_images[int(markerID)])
            src_h, src_w = src_image.shape[:2]

            # Scaling Code
            scale_factor = 15 / int(tvec[0][0][2])
            src_w, src_h = int(src_w * scale_factor), int(src_h * scale_factor)

            src_pts = np.array([[0, 0], [src_w, 0], [src_w , src_h], [0, src_h]])

            # Translation Code
#             if prevRvecX is not None:
#                 if abs(rvec[0][0][0]) > prevRvecX:
#                     src_pts += np.array([[0, -100], [0, -100], [0, -100], [0, -100]])
#                 elif abs(rvec[0][0][0]) < prevRvecX:
#                     src_pts += np.array([[0, 100], [0, 100], [0, 100], [0, 100]])
                    
#             if prevRvecY is not None:
#                 if rvec[0][0][1] > prevRvecY:
#                     src_pts += np.array([[-100, 0], [-100, 0], [-100, 0], [-100, 0]])
#                 elif rvec[0][0][1] < prevRvecX:
#                     src_pts += np.array([[100, 0], [100, 0], [100, 0], [100, 0]])
            
            crns = markerCorner.reshape([4, 2]).astype(int)

            H, _ = cv2.findHomography(srcPoints=src_pts, dstPoints=crns)
            augmented_image = cv2.warpPerspective(src_image, H, (w, h))

            cv2.fillConvexPoly(mask, crns, 255)
            frame = cv2.bitwise_and(frame, frame, mask=cv2.bitwise_not(mask))
            warp_image = cv2.bitwise_or(warp_image, augmented_image)

            cv2.putText(frame, f'X: {round(tvec[0][0][0],2)} Y: {round(tvec[0][0][1],2)} Z: {round(tvec[0][0][2],2)}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            rvec_degrees = np.degrees(rvec)
            cv2.putText(frame, str(rvec_degrees), (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            cv2.drawFrameAxes(frame, matrix_coefficients, distortion_coefficients, rvec, tvec, 2)  
        except IndexError as e:
            print(f"Error: {e}, marker ID: {int(markerID)}")
        

    # Combine the original frame with the augmented warp_image
    final_image = cv2.bitwise_or(frame, warp_image)

    return final_image

In [11]:
# Parameters
arucoDict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
arucoParams = cv2.aruco.DetectorParameters()

intrinsic_camera = np.array(((580.18421199, 0., 306.32847071), (0., 581.93758236, 241.22427157), (0., 0., 1. )))
distortion = np.array((-3.30325200e-01, 1.96909151e+00, -7.17910236e-03, -5.94455010e-04, -4.54516181e+00))

In [12]:
cap = cv2.VideoCapture(0)

prevRvecX = None
prevRvecY = None
frameCounter = 0

while cap.isOpened():
    _, img = cap.read() 
    
    corners, ids, rejected = cv2.aruco.detectMarkers(img, arucoDict, parameters=arucoParams)
    
    if len(corners) > 0:
        detected_markers = aruco_display(corners, ids, rejected, img)
        cv2.imshow("Detected Markers", detected_markers)
        
        wimage = float_image(img, corners, ids, rejected, intrinsic_camera, distortion)
        cv2.imshow("Wrap Image", wimage)
        
    cv2.imshow("Image", img)
    
    if cv2.waitKey(25) & 0xFF == 27:
        break
        
    frameCounter += 1
        
cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.7.0) D:\a\opencv-python\opencv-python\opencv\modules\objdetect\src\aruco\aruco_detector.cpp:860: error: (-215:Assertion failed) !_image.empty() in function 'cv::aruco::ArucoDetector::detectMarkers'


In [13]:
cap.release()
cv2.destroyAllWindows()

- X increase image shift to left
- Y increase image goes up

- If Y of rvec increase src_pts me x ka value kum kro
- If X of rvec increase src_pts me y ka value kum kro